In [11]:
import pandas as pd
import optuna
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score

In [ ]:
def objective(trial):
# Пространство поиска гиперпараметров
    params = {
        "max_depth": trial.suggest_int('max_depth', 3, 30),
        "min_samples_split": trial.suggest_int('min_samples_split', 2, 50),
        "min_samples_leaf": trial.suggest_int('min_samples_leaf', 1, 20),
        "criterion": trial.suggest_categorical('criterion', ['gini', 'entropy'])
    }

    # Задайте модель с гиперпараметрами из пространства поиска
    model = DecisionTreeClassifier(**params, random_state=42)

    # Укажите пайплайн
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
        ])
    
    # Укажите название метрики для объекта scorer 
    scorer = make_scorer(f1_score)

    # Задайте кросс-валидацию на 5 фолдах со scorer F1
    scores = cross_val_score(
        pipeline, # Пайплайн
        X_train, # Признаки
        y_train, # Целевая переменная
        cv=5, # 5-фолдовая кросс-валидация
        scoring=scorer # Передайте scorer 
        )
    
    return scores.mean()  # Верните среднее значение scores

In [ ]:
# Запустите исследование
# Фиксируем сид через семплер
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=25, show_progress_bar=True)

In [ ]:
# Лучшие гиперпараметры и качество
best_params = study.best_params
print("Лучшие гиперпараметры:", best_params)
best_value = study.best_value
print("Лучшее значение f1:", round(best_value, 4))

In [ ]:
# Визуализация результатов

fig1 = optuna.visualization.plot_optimization_history(study)  # Прогресс F1 по попыткам
fig2 = optuna.visualization.plot_param_importances(study)  # Важность гиперпараметров

plt.show()

In [ ]:
# Обучение финальной модели
best_model = DecisionTreeClassifier(**best_params, random_state=42)
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_model)
])

final_pipeline.fit(X_train, y_train) # Напишите ваш код для обучения пайплайна здесь 

y_pred = final_pipeline.predict(X_test)
print("Точность на тестовой выборке:", round(accuracy_score(y_test, y_pred), 4))
print("F1 на тестовой выборке:", round(f1_score(y_test, y_pred), 4))